# 03 · Trained classifier against an LLM pipeline

Kähler et al. (arXiv 2607.14882) found that the winner depends on the ruler: a trained model leads on agreement with past
human tags, and an LLM pipeline leads on experts' judgements of usefulness. That paper scores all labels as one flat set.
This notebook adds the tiers.

**Reduced forms.** Kähler's LLM-Ensemble uses several models and prompts. Here, keywords come from two models with one
prompt each, are mapped to labels by BGE-M3, and rated by one model, with the same final score: 0.3 × ensemble + 0.7 ×
relevance. Its score cut-off is chosen on 200 development items, as the trained model's label count is. Kähler's best trained model, XR-Transformer, has no build for this Mac, so the trained model is Omikuji, which
they also test. There is no expert grading here, so **this notebook can only measure agreement with existing tags, which
is the ruler that favoured trained models.**

**What would count as what, written before running.**
- Expected from Kähler: the trained model leads at exact labels on both test sets.
- Expected from Kähler: the LLM pipeline finds more rare and unseen labels than the trained model, whose recall on
  labels never seen in training is zero by construction.
- Open: whether the gap between them narrows at upper tiers. No prediction.
- Differences count only where the 95% paired bootstrap interval excludes zero.
- On HumSet, scores are shown both plainly and within each project's own labels.


In [1]:
import pandas as pd

from tiered_taxonomy import experiment as ex
from tiered_taxonomy import runs
from tiered_taxonomy.config import data_root, load_config

root = data_root()
config = load_config()
seed = config["seed"]
samples = config["samples"]


def sample_for(data):
    if data.name == ex.EURLEX:
        settings = samples["eurlex"]
        return ex.sample_eurlex(
            data,
            settings["normal_documents"],
            settings["long_documents"],
            settings["long_threshold_words"],
            seed,
        )
    return ex.sample_humset(data, samples["humset"]["per_language"], seed)


def table(results):
    rows = []
    for name, result in results.items():
        scores = result["scores"]
        row = {
            "method": name,
            "labels per item": scores["labels_per_item"],
            "exact F1": round(scores["exact"]["f1"], 3),
            "exact precision": round(scores["exact"]["precision"], 3),
            "exact recall": round(scores["exact"]["recall"], 3),
        }
        row.update({f"tier {tier} F1": round(value["f1"], 3) for tier, value in scores["tiers"].items()})
        row.update({f"{band} recall": value["recall"] for band, value in scores["rarity"].items()})
        if "run" in result:
            run = result["run"]
            row.update(
                {
                    "calls per item": round(run["calls"] / run["items"], 1),
                    "cost per 1000 items ($)": round(1000 * run["cost_usd"] / run["items"], 2),
                    "malformed": run["malformed"],
                    "invalid": run["invalid"],
                }
            )
        rows.append(row)
    return pd.DataFrame(rows)

/Users/matthewharris/git/nvidia-autoontology-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
eurlex = ex.load_eurlex(root)
humset = ex.load_humset(root)
benches = {data.name: runs.make_bench(data, root, config) for data in (eurlex, humset)}
sets = {data.name: data for data in (eurlex, humset)}
items = {name: sample_for(data) for name, data in sets.items()}
for name, bench in benches.items():
    runs.load_cards(bench)
    runs.embed_inputs(bench, items[name])
used = ex.project_labels([*humset.train, *humset.test], humset.taxonomy)
results = {}

In [3]:
from tiered_taxonomy.metrics import micro_f1, paired_difference

tuning = {}
for name, data in sets.items():
    bench = benches[name]
    sample = items[name]
    ids = [item.id for item in sample]
    tuning[name] = runs.tune_pipeline(
        bench, ex.pilot_subset(data.dev, config["pipeline"]["tuning_items"], seed)
    )
    group = {}
    for baseline in ("trained", "retrieval"):
        labels = ex.load_predictions(root, name, baseline, ids)
        group[baseline] = {"scores": ex.evaluate(data, sample, labels)}
    group["pipeline"] = runs.run_and_score(bench, "pipeline", sample)
    if name == ex.HUMSET:
        group["select_all"] = runs.run_and_score(bench, "select_all", sample)
    results[name] = group

## MultiEURLEX

In [4]:
table(results[ex.EURLEX])

,method,labels per item,exact F1,exact precision,exact recall,tier 1 F1,tier 2 F1,tier 3 F1,common recall,rare recall,unseen recall,calls per item,cost per 1000 items ($),malformed,invalid
0,trained,6.00,0.420,0.438,0.403,0.703,0.630,0.600,0.4369,0.0108,0.0000,NaN,NaN,NaN,NaN
1,retrieval,10.00,0.112,0.093,0.142,0.499,0.337,0.255,0.1429,0.0968,0.2000,NaN,NaN,NaN,NaN
2,pipeline,15.28,0.206,0.147,0.345,0.610,0.462,0.379,0.3398,0.3495,0.4917,3.0,0.87,0.0,0.0


## HumSet

In [5]:
table(results[ex.HUMSET])

,method,labels per item,exact F1,exact precision,exact recall,tier 1 F1,tier 2 F1,common recall,rare recall,calls per item,cost per 1000 items ($),malformed,invalid
0,trained,3.00,0.538,0.480,0.612,0.586,0.456,0.6158,0.3710,NaN,NaN,NaN,NaN
1,retrieval,7.37,0.110,0.073,0.227,0.231,0.072,0.2221,0.5161,NaN,NaN,NaN,NaN
2,pipeline,2.19,0.374,0.388,0.361,0.444,0.213,0.3614,0.3387,3.0,0.33,1.0,0.0
3,select_all,2.06,0.405,0.434,0.380,0.462,0.282,0.3804,0.3548,1.0,0.15,0.0,0.0


## Are the differences real?

Trained minus pipeline, at each tier, with 95% intervals from resampling the same items for both.

In [6]:
differences = {}
for name, data in sets.items():
    sample = items[name]
    ids = [item.id for item in sample]
    trained = ex.load_predictions(root, name, "trained", ids)
    pipeline = ex.load_predictions(root, name, "pipeline", ids)
    rows = {"exact": paired_difference(micro_f1, ex.gold_exact(sample), trained, pipeline)}
    for tier in data.tiers:
        rolled_t = [data.taxonomy.roll_up(labels, tier) for labels in trained]
        rolled_p = [
            data.taxonomy.roll_up({label for label in labels if label in data.taxonomy.nodes}, tier)
            for labels in pipeline
        ]
        rows[f"tier {tier}"] = paired_difference(micro_f1, ex.gold_at_tier(sample, tier), rolled_t, rolled_p)
    differences[name] = {key: [round(value, 3) for value in row] for key, row in rows.items()}
pd.DataFrame(
    {
        (name, key): {"difference": row[0], "low": row[1], "high": row[2]}
        for name, group in differences.items()
        for key, row in group.items()
    }
).T

difference    low   high
eurlex exact        0.214  0.196  0.231
       tier 1       0.093  0.074  0.111
       tier 2       0.167  0.147  0.189
       tier 3       0.222  0.197  0.246
humset exact        0.193  0.177  0.208
       tier 1       0.142  0.126  0.159
       tier 2       0.243  0.216  0.267

## HumSet: within each project's own labels

A label a project never used cannot be a mistake in that project, so this removes those from the scoring.

In [7]:
within = {}
sample = items[ex.HUMSET]
ids = [item.id for item in sample]
for method in results[ex.HUMSET]:
    labels = ex.load_predictions(root, ex.HUMSET, method, ids)
    within[method] = ex.evaluate_within_project(humset, sample, labels, used)
pd.DataFrame(
    {
        method: {
            f"tier {tier} {part}": round(values[part], 3)
            for tier, values in tiers.items()
            for part in ("precision", "recall", "f1")
        }
        for method, tiers in within.items()
    }
).T

,tier 1 precision,tier 1 recall,tier 1 f1,tier 2 precision,tier 2 recall,tier 2 f1
trained,0.527,0.670,0.590,0.429,0.522,0.471
retrieval,0.174,0.413,0.245,0.062,0.213,0.097
pipeline,0.485,0.434,0.458,0.349,0.172,0.231
select_all,0.540,0.425,0.476,0.359,0.278,0.313


## Rare labels, compared fairly

The pipeline returns about 15 labels per law and the trained classifier 6, so more recall is partly just more guesses. This
gives the trained classifier the same number of labels, from its saved rankings.

In [8]:
import json

from tiered_taxonomy.metrics import top_k

sample = items[ex.EURLEX]
ranked = json.loads(runs.trained_ranked_path(root, ex.EURLEX).read_text(encoding="utf-8"))
same_count = round(results[ex.EURLEX]["pipeline"]["scores"]["labels_per_item"])
trained_more = [top_k(ranked[item.id], same_count) for item in sample]
trained_same_count = ex.evaluate(eurlex, sample, trained_more)
{
    "labels per law": same_count,
    "rarity": trained_same_count["rarity"],
    "exact precision": round(trained_same_count["exact"]["precision"], 3),
}

{'labels per law': 15,
 'rarity': {'common': {'recall': 0.6031, 'gold_labels': 3605},
  'rare': {'recall': 0.0538, 'gold_labels': 186},
  'unseen': {'recall': 0.0, 'gold_labels': 120}},
 'exact precision': 0.243}

In [9]:
ex.save_results(
    "03_bakeoff",
    {
        "results": results,
        "differences": differences,
        "humset_within_project": within,
        "pipeline_tuning": tuning,
        "models": config["models"],
        "trained_same_count": trained_same_count,
        "same_count": same_count,
    },
);